In [1]:
%load_ext cudf.pandas

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
sns.set()

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
df = pd.read_csv("/home/dias-benchmarks/notebooks/saisandeepjallepalli/adidas-retail-eda-data-visualization/input/adidas-us-retail-products-dataset/adidas.csv")
df = pd.concat([df]*factor)
df.info()

In [4]:
### cell 0 
_ = df.isna().sum()

In [5]:
### cell 1
df.dropna(inplace=True, axis=0)

In [6]:
### cell 2 (7 GPU, 1 CPU)
df['original_price'] = df['original_price'].str.split('$')
df['original_price'] = df['original_price'].str[1]

In [7]:
### cell 3 (all on GPU)
df['selling_price'].sort_values(ascending=False)[:30].values

array([240, 200, 196, 180, 180, 180, 180, 180, 180, 180, 180, 180, 144,
       144, 144, 135, 135, 128, 128, 128, 128, 126, 126, 126, 126, 120,
       120, 112, 112, 112])

In [8]:
### cell 4 (all on GPU)
df['original_price'] = df['original_price'].astype('int64')

In [9]:
### cell 5 (all on GPU)
df['Discount(%)'] = round(((df['original_price'] - df['selling_price']) / (df['original_price']))*100,2)

In [10]:
%%time
### cell 6 (4 GPU, 1 CPU repr_latex)
_ = df['Discount(%)'].value_counts().head(15)

Discount(%)
20.00    454
30.00    173
10.00     44
29.23     20
28.57     15
28.89     15
17.86     11
40.00     11
27.78      8
29.41      8
28.00      7
18.18      6
8.89       5
29.33      5
7.14       4
Name: count, dtype: int64

                                                                                                                
                                           Total time elapsed: 0.482 seconds                                    
                                         4 GPU function calls in 0.062 seconds                                  
                                         1 CPU function calls in 0.007 seconds                                  
                                                                                                                
                                                         Stats                                                  
                                                                                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function                   ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.__getitem__      │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ IndexOpsMixin.value_counts │ 1          │ 0.039       │ 0.039       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.head               │ 1          │ 0.003       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ Series.__repr__            │ 1          │ 0.017       │ 0.017       │ 0          │ 0.000       │ 0.000       │
│ NDFrame._repr_latex_       │ 0          │ 0.000       │ 0.000       │ 1          │ 0.007       │ 0.007       │
└────────────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- NDFrame._repr_latex_

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=96459;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.